# Demo of Classification

This notebook demonstrates the code necessary to train different classifiers and use these to predict new MSMS spectra of either being of "interest" or "other". 

## Setup

In [ ]:
from AnnoMe.Classification import (
    generate_embeddings,
    add_all_metadata,
    show_dataset_overview,
    generate_summary,
    train_and_classify,
    generate_prediction_overview,
    generate_ml_metrics_overview,
    set_random_seeds,
)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold

from collections import OrderedDict
import os
import re

set_random_seeds(42)

## Parameters

In [ ]:
# fmt: off
# parameters
# 

# Output directory
output_dir = f"./output/IsoFlavonoids_PublicDBs/"

# Main datasets to process for classification
max_instances = 10000
datasets = []

# Iterate through the folder and find matching files
base_folder = "../resources/libraries_filtered/IsoFlavonoids/"
for typ, suffix in {"train - relevant": "^(MSnLib_.*)___iso_and_flavonoids__MatchingSmiles\\.mgf$",
                    "train - other": "^(MSnLib_.*)___NonMatchingSmiles\\.mgf$"}.items():
    for file_name in os.listdir(base_folder):
        match = re.match(suffix, file_name)
        if match:
            ty = " gt other" if typ == "train - other" else " gt relevant"
            datasets.append({
                "name": f"{match.group(1)} {ty}",
                "type": typ,
                "file": os.path.join(base_folder, file_name),
                "fragmentation_method": "fragmentation_method",
                "colour": "#80BF02",  # Assign a unique color for this type
                "randomly_sample": (max_instances, 42)
            })
            print(f"   - '{match.group(1)}' in '{file_name}', using as '{typ}'")

datasets.extend([
    ## MS/MS of BOKU iBAM
    {
        "name": "BOKU iBAM - gt relevant",
        "type": "validation - relevant",
        "file": os.path.join("..", "resources", "libraries_filtered", "IsoFlavonoids", "BOKU_iBAM___iso_and_flavonoids__MatchingSmiles.mgf"),
        "fragmentation_method": "fragmentation_method",
        "colour": "#80BF02",
    },
    {
        "name": "BOKU iBAM - gt other",
        "type": "validation - other",
        "file": os.path.join("..", "resources", "libraries_filtered", "IsoFlavonoids", "BOKU_iBAM___NonMatchingSmiles.mgf"),
        "fragmentation_method": "fragmentation_method",
        "colour": "#80BF02",
    },
])

# meta-data to add to the output from the MS/MS spectra
data_to_add = OrderedDict(
    [
        ("name", ["feature_id", "name", "title", "compound_name"]),
        ("formula", ["formula"]),
        ("smiles", ["smiles"]),
        ("adduct", ["adduct", "precursor_type"]),
        ("ionMode", ["ionmode"]),
        ("RTINSECONDS", ["rtinseconds", "retention_time"]),
        ("precursor_mz", ["pepmass", "precursor_mz"]),
        ("fragmentation_method", ["fragmentation_method", "fragmentation_mode"]),
        ("CE", ["collision_energy"]),
    ]
)

training_subsets = {
    "hcd_pos_20.0" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "positive") & (x["CE"] == "20.0"),
    "hcd_pos_30.0" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "positive") & (x["CE"] == "30.0"),
    "hcd_pos_60.0" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "positive") & (x["CE"] == "60.0"),
    "hcd_pos_step[20,45,70]" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "positive") & (x["CE"] in ["45.0", "stepped20,45,70ev(absolute)"]),
    
    "hcd_neg_20.0" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "negative") & (x["CE"] == "20.0"),
    "hcd_neg_30.0" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "negative") & (x["CE"] == "30.0"),
    "hcd_neg_60.0" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "negative") & (x["CE"] == "60.0"),
    "hcd_neg_step[20,45,70]" : lambda x: (x["fragmentation_method"] == "hcd") & (x["ionMode"] == "negative") & (x["CE"] in ["45.0", "stepped20,45,70ev(absolute)"]),
}

# derived, do not change
colors = {ds["name"]: ds["colour"] for ds in datasets}
# fmt: on

## Execute pipeline

In [ ]:
# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

df = generate_embeddings(datasets, data_to_add, n_jobs=8)

# add associated metadata
df = add_all_metadata(datasets, df)

classifiers_to_compare = {
    "default_set": {
        "classifiers": {"LDA": LinearDiscriminantAnalysis(solver="svd", store_covariance=True, n_components=1)},
        "cross_validation": StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
        "min_prediction_threshold": 120,
        "description": "Default set of diverse classifiers for comparison",
    }
}

# iterate over the training subsets, produces better output
for subset_name in training_subsets:
    print(f"Processing subset: {subset_name}")
    print(f"##############################################################################")

    # Get the subset function
    subset_fn = training_subsets[subset_name]

    # Create output directory for the subset
    c_output_dir = f"{output_dir}/subset_{subset_name}/"
    os.makedirs(c_output_dir, exist_ok=True)

    # subset the dataframe
    df_subset = df[df.apply(subset_fn, axis=1)].reset_index(drop=True)

    # train and predict new datasets
    df_train, df_validation, df_inference, df_metrics, trained_classifiers = train_and_classify(df_subset, classifiers_to_compare=classifiers_to_compare, subsets={subset_name: subset_fn})
    generate_prediction_overview(df_subset, df_train, c_output_dir, "training", min_prediction_threshold=10)
    generate_prediction_overview(df_subset, df_validation, c_output_dir, "validation", min_prediction_threshold=10)
    generate_prediction_overview(df_subset, df_inference, c_output_dir, "inference", min_prediction_threshold=10)

    # Generate an overview of the machine learning metrics
    generate_ml_metrics_overview(df_metrics, c_output_dir)

In [ ]:
generate_summary(output_dir)